In [4]:
import json, os, random, statistics, time, unicodedata
import requests

# ============================================================================
# CONFIG
# ============================================================================
BASE = "https://hackathon.api.qloo.com"
SECRET_NAME = "qloo_hack"
THROTTLE_SEC = 0.35
DEBUG_DUMP = True
CHUNK = 25                      # brands per panel call (avoids truncation/URL limits)
STABILITY_REPEATS = 5
OUT_DIR = os.path.join(os.getcwd(), "qloo_validation")

def load_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(SECRET_NAME)
    except Exception:
        key = os.environ.get("QLOO_API_KEY") or os.environ.get("qloo_hack")
        if key: return key
        raise RuntimeError("No API key. Kaggle: attach secret 'qloo_hack' + enable Internet.")

KEY = load_api_key()
SESSION = requests.Session()
SESSION.headers.update({"X-Api-Key": KEY, "Accept": "application/json"})

# ============================================================================
# API CLIENT
# ============================================================================
_LAST = 0.0
STATS = {"calls": 0, "latency": [], "status": {}, "dumps": set()}

def _dump(tag, obj, limit=2500):
    """Print the first raw response per endpoint tag so schema surprises are visible."""
    if not DEBUG_DUMP or tag in STATS["dumps"]: return
    STATS["dumps"].add(tag)
    txt = json.dumps(obj, indent=2, ensure_ascii=False) if obj is not None else "None"
    print(f"\n--- RAW [{tag}] ---\n{txt[:limit]}{'...[trunc]' if len(txt) > limit else ''}\n--- END ---\n")

def qloo_get(path, params=None, tag="", quiet=False):
    """GET with throttle + retry on 429/5xx. Returns (status, json|None)."""
    global _LAST
    params = {k: v for k, v in (params or {}).items() if v not in (None, "")}
    backoff = 2.0
    for _ in range(6):
        gap = time.monotonic() - _LAST
        if gap < THROTTLE_SEC: time.sleep(THROTTLE_SEC - gap + random.uniform(0, 0.1))
        _LAST = time.monotonic()
        STATS["calls"] += 1
        t0 = time.monotonic()
        try:
            r = SESSION.get(BASE + path, params=params, timeout=30)
        except requests.RequestException as e:
            print(f"  [network] {path}: {e}"); time.sleep(backoff); backoff *= 2; continue
        STATS["latency"].append(time.monotonic() - t0)
        STATS["status"][r.status_code] = STATS["status"].get(r.status_code, 0) + 1
        if r.status_code == 200:
            try: data = r.json()
            except ValueError: return 200, None
            _dump(tag or path, data); return 200, data
        if r.status_code == 429:
            print(f"  [429] sleeping {backoff:.0f}s"); time.sleep(backoff); backoff = min(backoff * 2, 60); continue
        if r.status_code in (500, 502, 503, 504):
            time.sleep(backoff); backoff *= 2; continue
        if r.status_code == 401:
            raise RuntimeError("401: base must be hackathon.api.qloo.com, key in X-Api-Key header.")
        if not quiet:
            print(f"  [{r.status_code}] {path} {str(params)[:160]} :: {r.text[:160]}")
        return r.status_code, None
    return None, None

# ============================================================================
# SAMPLES
# ============================================================================
CELEBRITIES = [
    # comedians / actors / creators
    "Cedric the Entertainer", "Kevin Hart", "Dave Chappelle", "Chris Rock", "Ali Wong",
    "Trevor Noah", "Will Smith", "Denzel Washington", "Samuel L. Jackson", "Jennifer Aniston",
    "Ryan Reynolds", "George Clooney", "Dwayne Johnson", "Millie Bobby Brown", "Selena Gomez",
    "Hailey Bieber", "Shay Mitchell", "Tracee Ellis Ross", "Jessica Alba", "Zendaya",
    "Kendall Jenner", "Kristen Bell",
    # musicians
    "Rihanna", "Beyonce", "Taylor Swift", "Drake", "Snoop Dogg", "Kendrick Lamar", "Jay-Z",
    "Lady Gaga", "Billie Eilish", "Ariana Grande", "Post Malone", "The Weeknd", "Bad Bunny",
    "Bruno Mars", "SZA", "Doja Cat", "Katy Perry", "Adele", "Ed Sheeran",
    # athletes
    "LeBron James", "Michael Jordan", "Serena Williams", "Stephen Curry", "Naomi Osaka",
]
BRANDS_MAIN = [
    "Nike", "Adidas", "Pepsi", "Coca-Cola", "Sprite", "Diet Coke", "Starbucks", "McDonald's",
    "Apple", "Samsung", "Gucci", "Louis Vuitton", "Supreme", "Patagonia", "The North Face",
    "Lululemon", "Red Bull", "Heineken", "Corona Extra", "Modelo Especial", "Jack Daniel's",
    "Hennessy", "Jameson", "Patron", "Casamigos", "Aviation American Gin", "Fenty Beauty",
    "Sephora", "Glossier", "MAC Cosmetics", "Old Spice", "Netflix", "Spotify", "Airbnb",
    "Uber", "Chipotle", "Chick-fil-A", "In-N-Out",
]
CONTROL_BRANDS = [
    "Smartwater", "Rare Beauty", "Rhode", "Florence by Mills", "Pattern Beauty", "Beis",
    "De Soi", "Honest", "818 Tequila", "Haus Labs", "Indoggo Gin", "Maison No. 9",
    "Gran Coramino", "Hello Bello", "Teremana",
]
# documented real relationships: (celebrity, brand, kind)  kind = founder | endorser
POSITIVE_PAIRS = [
    ("Ryan Reynolds", "Aviation American Gin", "founder"), ("George Clooney", "Casamigos", "founder"),
    ("Dwayne Johnson", "Teremana", "founder"), ("Rihanna", "Fenty Beauty", "founder"),
    ("Selena Gomez", "Rare Beauty", "founder"), ("Hailey Bieber", "Rhode", "founder"),
    ("Millie Bobby Brown", "Florence by Mills", "founder"), ("Tracee Ellis Ross", "Pattern Beauty", "founder"),
    ("Shay Mitchell", "Beis", "founder"), ("Katy Perry", "De Soi", "founder"),
    ("Jessica Alba", "Honest", "founder"), ("Kendall Jenner", "818 Tequila", "founder"),
    ("Lady Gaga", "Haus Labs", "founder"), ("Snoop Dogg", "Indoggo Gin", "founder"),
    ("Post Malone", "Maison No. 9", "founder"), ("Kevin Hart", "Gran Coramino", "founder"),
    ("Kristen Bell", "Hello Bello", "founder"),
    ("Jennifer Aniston", "Smartwater", "endorser"), ("Michael Jordan", "Nike", "endorser"),
    ("LeBron James", "Nike", "endorser"), ("Serena Williams", "Nike", "endorser"),
    ("Naomi Osaka", "Nike", "endorser"), ("Beyonce", "Pepsi", "endorser"),
    ("Taylor Swift", "Diet Coke", "endorser"), ("Drake", "Sprite", "endorser"),
]
DOC = {(c, b) for c, b, _ in POSITIVE_PAIRS}

# ============ EXPANSION: more mid-size brands, mid-tier + famous celebrities ============
NEW_PAIRS = [
    # --- founder / owner: spirits & drinks ---
    ("Justin Timberlake","Sauza 901","founder"), ("Bob Dylan","Heaven's Door","founder"),
    ("Drake","Virginia Black","founder"), ("Bryan Cranston","Dos Hombres","founder"),
    ("Aaron Paul","Dos Hombres","founder"), ("Nick Jonas","Villa One","founder"),
    ("LeBron James","Lobos 1707","founder"), ("Sammy Hagar","Santo Tequila","founder"),
    ("Guy Fieri","Santo Tequila","founder"), ("Darius Rucker","Backstage Whiskey","founder"),
    ("Willie Nelson","Old Whiskey River","founder"), ("Metallica","Blackened American Whiskey","founder"),
    ("Jay-Z","D'Usse","founder"), ("Kate Hudson","King St. Vodka","founder"),
    ("Rita Ora","Prospero Tequila","founder"), ("Brad Pitt","Miraval","founder"),
    ("Angelina Jolie","Miraval","founder"), ("Bruno Mars","Selvarey Rum","founder"),
    ("Conor McGregor","Proper No. Twelve","founder"), ("Mark Wahlberg","Flecha Azul","founder"),
    ("Michael Jordan","Cincoro","founder"), ("Eva Longoria","Casa Del Sol","founder"),
    ("Cardi B","Whipshots","founder"), ("Michael B. Jordan","J'Ouvert","founder"),
    ("David Beckham","Haig Club","founder"), ("Nina Dobrev","Fresh Vine Wine","founder"),
    ("Julianne Hough","Fresh Vine Wine","founder"), ("Cameron Diaz","Avaline","founder"),
    ("Blake Lively","Betty Buzz","founder"), ("Emma Chamberlain","Chamberlain Coffee","founder"),
    ("Bella Hadid","Kin Euphorics","founder"), ("Dwayne Johnson","ZOA Energy","founder"),
    ("Ryan Reynolds","Mint Mobile","founder"),
    # --- founder / owner: beauty, fashion, lifestyle, tech ---
    ("Ariana Grande","r.e.m. beauty","founder"), ("Harry Styles","Pleasing","founder"),
    ("Gwyneth Paltrow","Goop","founder"), ("Kylie Jenner","Kylie Cosmetics","founder"),
    ("Kim Kardashian","SKIMS","founder"), ("Dr. Dre","Beats by Dre","founder"),
    ("Jennifer Lopez","JLo Beauty","founder"), ("Alicia Keys","Keys Soulcare","founder"),
    ("Gwen Stefani","GXVE Beauty","founder"), ("Pharrell Williams","Humanrace","founder"),
    ("Reese Witherspoon","Draper James","founder"), ("Kourtney Kardashian","Poosh","founder"),
    ("Victoria Beckham","Victoria Beckham Beauty","founder"), ("Tom Brady","TB12","founder"),
    ("Naomi Osaka","Kinlo","founder"), ("Lizzo","Yitty","founder"),
    ("Rihanna","Savage X Fenty","founder"), ("Jessica Alba","The Honest Company","founder"),
    # --- endorser / ambassador ---
    ("Jennifer Aniston","Aveeno","endorser"), ("Zendaya","Lancome","endorser"),
    ("Zendaya","Valentino","endorser"), ("Timothee Chalamet","Chanel","endorser"),
    ("Margot Robbie","Chanel","endorser"), ("Megan Thee Stallion","Revlon","endorser"),
    ("Cardi B","Reebok","endorser"), ("Kendrick Lamar","Reebok","endorser"),
    ("Stephen Curry","Under Armour","endorser"), ("Dwayne Johnson","Under Armour","endorser"),
    ("Patrick Mahomes","Oakley","endorser"), ("Serena Williams","Gatorade","endorser"),
    ("David Beckham","H&M","endorser"),
]

# celebrities with no encoded deal: extra rows for the matrix / z-scores / spotlight
EXTRA_CELEBS = [
    "Awkwafina", "Kenan Thompson", "Tiffany Haddish", "Anthony Anderson", "Jamie Foxx",
    "Idris Elba", "Issa Rae", "Steve Harvey", "Terry Crews", "Ice Cube", "Ludacris", "T.I.",
    "Common", "Queen Latifah", "Regina Hall", "Taraji P. Henson", "Gabrielle Union",
    "Anthony Mackie", "John Legend", "Chrissy Teigen", "Questlove", "Usher", "Janelle Monae",
    "Lil Nas X", "Olivia Rodrigo", "Sabrina Carpenter", "Tate McRae", "Chappell Roan",
    "Zach Bryan", "Luke Combs", "Morgan Wallen", "Kacey Musgraves", "Jason Momoa",
    "Pedro Pascal", "Jenna Ortega", "Sydney Sweeney", "Florence Pugh", "Paul Mescal",
    "Jacob Elordi", "Simu Liu", "Travis Kelce", "Caitlin Clark", "Simone Biles",
    "Joe Burrow", "Shohei Ohtani", "Jayson Tatum", "Mike Epps", "Method Man",
]

# mid-size brands across categories (extra non-deal columns for the matrix)
MID_BRANDS = [
    # spirits / beer / wine
    "Tito's Handmade Vodka", "Grey Goose", "Ketel One", "Hendrick's Gin", "Bombay Sapphire",
    "Tanqueray", "Don Julio", "Jose Cuervo", "Espolon", "Maker's Mark", "Woodford Reserve",
    "Bulleit", "Jim Beam", "Fireball", "Johnnie Walker", "Glenfiddich", "Bacardi",
    "Captain Morgan", "Malibu", "Baileys", "Aperol", "Dos Equis", "Blue Moon", "Sierra Nevada",
    "Samuel Adams", "Stella Artois", "Guinness", "White Claw", "Truly", "Barefoot",
    # beauty / personal care
    "Drunk Elephant", "The Ordinary", "CeraVe", "Tatcha", "Kiehl's", "Clinique", "Urban Decay",
    "NARS", "Benefit Cosmetics", "Too Faced", "Tarte", "e.l.f. Cosmetics", "NYX Professional Makeup",
    "Maybelline", "Olaplex", "Aveda", "Dove", "Neutrogena", "La Roche-Posay", "Sol de Janeiro",
    "Summer Fridays", "Bumble and bumble",
    # apparel / footwear / outdoor
    "Carhartt", "Levi's", "Dickies", "Vans", "Converse", "Champion", "Under Armour", "New Balance",
    "Reebok", "Puma", "Fila", "Timberland", "Dr. Martens", "Ugg", "Crocs", "Birkenstock", "Hoka",
    "On Running", "Allbirds", "Stussy", "Kith", "Aritzia", "Madewell", "J.Crew", "Zara", "H&M",
    "Uniqlo", "Urban Outfitters", "Anthropologie", "Free People", "Fashion Nova", "Oakley",
    "Ray-Ban", "Warby Parker", "Valentino", "Lancome", "Chanel", "Dior",
    # food / drink / restaurants
    "Celsius", "Monster Energy", "Liquid Death", "Prime Hydration", "Vita Coco", "Topo Chico",
    "Dunkin'", "Panera Bread", "Shake Shack", "Sweetgreen", "Taco Bell", "Popeyes", "Wingstop",
    "Cheetos", "Doritos", "Oreo", "Skittles", "Ben & Jerry's", "Oatly", "Chobani", "Clif Bar",
    "Gatorade", "Aveeno",
    # tech / home / auto / lifestyle
    "Peloton", "Fitbit", "GoPro", "Bose", "Sonos", "Beats by Dre", "JBL", "Dyson", "Yeti",
    "Stanley", "Hydro Flask", "Casper", "Harley-Davidson", "Jeep", "Tesla", "Ford", "Hyundai",
]

def _dedupe(seq): return list(dict.fromkeys(seq))
_have = {(c, b) for c, b, _ in POSITIVE_PAIRS}
POSITIVE_PAIRS += [p for p in NEW_PAIRS if (p[0], p[1]) not in _have]
CELEBRITIES = _dedupe(CELEBRITIES + [c for c, _, _ in NEW_PAIRS] + EXTRA_CELEBS)
CONTROL_BRANDS = _dedupe(CONTROL_BRANDS + [b for _, b, _ in POSITIVE_PAIRS] + MID_BRANDS)
CONTROL_BRANDS = [b for b in CONTROL_BRANDS if b not in BRANDS_MAIN]
# ============ END EXPANSION ============

DOC_BY_CELEB = {}
for c, b, _ in POSITIVE_PAIRS: DOC_BY_CELEB.setdefault(c, set()).add(b)

# ============================================================================
# PARSING HELPERS
# ============================================================================
def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return " ".join(s.casefold().split())

def similarity(a, b):
    a, b = norm(a), norm(b)
    if not a or not b: return 0.0
    if a == b or a in b or b in a: return 1.0
    ta, tb = set(a.split()), set(b.split())
    return round(0.5 * len(ta & tb) / max(1, len(ta | tb)) +
                 0.5 * (1 - abs(len(a) - len(b)) / max(len(a), len(b))), 4)

def _entity_fields(item):
    e = item.get("entity") if isinstance(item, dict) else None
    e = e if isinstance(e, dict) else {}
    eid = item.get("entity_id") or item.get("id") or item.get("urn") or e.get("entity_id") or e.get("id")
    name = item.get("name") or e.get("name") or item.get("title") or ""
    etype = item.get("type") or e.get("type") or item.get("entity_type") or ""
    if not etype:
        types = item.get("types") or e.get("types") or []
        etype = " ".join(types) if isinstance(types, list) else str(types)
    aka = item.get("akas") or e.get("akas") or []
    return eid, name, etype, ([aka] if isinstance(aka, str) else aka)

def iter_result_items(payload):
    """Yield items from search (results=list) or insights (results.entities=list)."""
    if not isinstance(payload, dict): return
    r = payload.get("results")
    if isinstance(r, dict):
        for k in ("entities", "items", "results", "data", "audiences", "tags"):
            if isinstance(r.get(k), list):
                for it in r[k]: yield it
    elif isinstance(r, list):
        for it in r: yield it

def extract_score(item):
    """FIXED: Qloo puts the number at item['query']['affinity']."""
    if not isinstance(item, dict): return None
    q = item.get("query")
    if isinstance(q, dict):
        v = q.get("affinity")
        if isinstance(v, (int, float)) and not isinstance(v, bool): return float(v)
    for key in ("affinity", "score", "similarity"):
        v = item.get(key)
        if isinstance(v, (int, float)) and not isinstance(v, bool): return float(v)
    return None

# ============================================================================
# RESOLUTION  (/search?query=...)
# ============================================================================
_RES = {}
def resolve(name, required_type=""):
    """Best fuzzy-matching entity; if required_type set, other types are rejected."""
    key = f"{norm(name)}|{required_type}"
    if key in _RES: return _RES[key]
    best = None
    st, data = qloo_get("/search", {"query": name}, tag="search")
    if st == 200 and data:
        for it in iter_result_items(data):
            eid, ename, etype, akas = _entity_fields(it)
            if not eid or not ename: continue
            sim = max([similarity(name, ename)] + [similarity(name, a) for a in akas if a])
            if sim < 0.55: continue
            if required_type and norm(required_type) not in norm(etype): continue
            if best is None or sim > best["sim"]:
                best = {"id": eid, "name": ename, "type": etype, "sim": sim}
    _RES[key] = best
    return best

def resolve_signal(name, probe_brand_id):
    """Try every person/artist candidate until one actually works as a taste signal."""
    st, data = qloo_get("/search", {"query": name}, tag="search")
    cands = []
    for it in (iter_result_items(data) if st == 200 and data else []):
        eid, ename, etype, akas = _entity_fields(it)
        if not eid or not ename: continue
        sim = max([similarity(name, ename)] + [similarity(name, a) for a in akas if a])
        if sim >= 0.85 and ("person" in norm(etype) or "artist" in norm(etype)):
            cands.append((sim, eid, ename, etype))
    for sim, eid, ename, etype in sorted(cands, reverse=True):
        if fit_score(eid, probe_brand_id) is not None:
            return {"id": eid, "name": ename, "type": etype, "sim": sim}
    return None

def resolve_all(kind, names, required_type=""):
    out, miss = {}, []
    print(f"\n[RESOLVE] {kind}: {len(names)}")
    for n in names:
        e = resolve(n, required_type)
        if e: out[n] = e
        else: miss.append(n)
    print(f"[RESOLVE] {kind}: {len(out)}/{len(names)} resolved. missing: {miss}")
    return out, miss

# ============================================================================
# FIT SCORE PRIMITIVES
# ============================================================================
def fit_score(celeb_id, brand_id, extra=None):
    """Single pair: affinity of brand toward celebrity's taste (filter.results.entities)."""
    p = {"filter.type": "urn:entity:brand", "signal.interests.entities": celeb_id,
         "filter.results.entities": brand_id, "take": 1}
    p.update(extra or {})
    st, data = qloo_get("/v2/insights", p, tag="insights[fit]")
    if st != 200 or not data: return None
    items = list(iter_result_items(data))
    return extract_score(items[0]) if items else None

def panel_scores(celeb_id, brand_ids):
    """One celebrity vs MANY brands, chunked. Missing brands are simply absent (never filled)."""
    out = {}
    for i in range(0, len(brand_ids), CHUNK):
        p = {"filter.type": "urn:entity:brand", "signal.interests.entities": celeb_id,
             "filter.results.entities": ",".join(brand_ids[i:i + CHUNK]), "take": 50}
        st, data = qloo_get("/v2/insights", p, tag="insights[panel]")
        if st != 200 or not data: continue
        for it in iter_result_items(data):
            eid, _, _, _ = _entity_fields(it)
            s = extract_score(it)
            if eid and s is not None: out[eid] = s
    return out

# ============================================================================
# STATS
# ============================================================================
def auc(pos, neg):
    """P(random positive > random negative), ties = 0.5."""
    if not pos or not neg: return None
    allv = sorted([(v, 1) for v in pos] + [(v, 0) for v in neg], key=lambda x: x[0])
    i, rs = 0, 0.0
    while i < len(allv):
        j = i
        while j + 1 < len(allv) and allv[j + 1][0] == allv[i][0]: j += 1
        for k in range(i, j + 1):
            if allv[k][1]: rs += (i + j) / 2.0 + 1.0
        i = j + 1
    n1, n0 = len(pos), len(neg)
    return (rs - n1 * (n1 + 1) / 2.0) / (n1 * n0)

def pct_rank(v, ref):
    if not ref: return None
    return (sum(1 for r in ref if r < v) + 0.5 * sum(1 for r in ref if r == v)) / len(ref)

def verdict(x, hi=0.75, mid=0.60):
    return "n/a" if x is None else "PASS" if x >= hi else "WARN" if x >= mid else "FAIL"

def banner(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

# ============================================================================
# MAIN
# ============================================================================
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    t_start = time.time()

    banner("STEP 0 - HEALTH CHECK")
    st, _ = qloo_get("/search", {"query": "Nike"}, tag="search[health]")
    if st != 200: print("Health check failed."); return
    print("API OK")

    # ---------------- GATE 1: resolution ----------------
    banner("GATE 1 - ENTITY RESOLUTION")
    brands, _ = resolve_all("mainstream brands", BRANDS_MAIN, "brand")
    controls, _ = resolve_all("control brands", CONTROL_BRANDS, "brand")
    probe = brands["Nike"]["id"]
    celebs, miss = {}, []
    for n in CELEBRITIES:
        e = resolve_signal(n, probe)
        if e: celebs[n] = e
        else: miss.append(n)
    print(f"celebrities valid as signal: {len(celebs)}/{len(CELEBRITIES)}; missing: {miss}")
    allb = {**brands, **controls}
    gate1 = len(celebs) / len(CELEBRITIES) >= 0.70 and len(brands) / len(BRANDS_MAIN) >= 0.85
    print(f"GATE 1 -> {'PASS' if gate1 else 'WARN'} ({len(celebs)} celebs, {len(brands)} main brands, {len(controls)} control brands)")
    if "Cedric the Entertainer" not in celebs: print("!! Cedric did not resolve")

    # ---------------- build the full celebrity x brand matrix ----------------
    banner("GATE 2 - FULL CELEBRITY x BRAND MATRIX (chunked panel calls)")
    id2b = {e["id"]: n for n, e in allb.items()}
    print("brand name collisions (same entity id):", [n for n, e in allb.items() if id2b[e["id"]] != n])
    bids = list(id2b.keys())
    M = {}                                             # M[celeb][brand] = affinity (absent = no score)
    attempted = scored = 0
    for cname, cent in celebs.items():
        sc = panel_scores(cent["id"], bids)
        M[cname] = {id2b[k]: v for k, v in sc.items() if k in id2b}
        attempted += len(bids); scored += len(M[cname])
        print(f"   {cname:<24} {len(M[cname])}/{len(bids)} brands scored")
    coverage = scored / max(1, attempted)
    allvals = [v for r in M.values() for v in r.values()]
    std = statistics.pstdev(allvals) if len(allvals) > 2 else 0.0
    gate2 = coverage >= 0.75
    print(f"GATE 2 -> {'PASS' if gate2 else 'FAIL'} coverage {100 * coverage:.0f}% ({scored}/{attempted}); "
          f"min={min(allvals):.3f} max={max(allvals):.3f} mean={statistics.mean(allvals):.3f} std={std:.3f}")

    # per-brand z-score across celebrities (removes brand-popularity baseline)
    colvals = {}
    for c, row in M.items():
        for b, v in row.items(): colvals.setdefault(b, []).append(v)
    colstat = {b: (statistics.mean(v), statistics.pstdev(v)) for b, v in colvals.items() if len(v) >= 8}
    Z = {c: {b: (v - colstat[b][0]) / colstat[b][1] for b, v in row.items()
             if b in colstat and colstat[b][1] > 1e-9} for c, row in M.items()}

    # ---------------- GATE 3: discrimination ----------------
    banner("GATE 3 - DO DOCUMENTED DEALS RANK ABOVE NON-DEALS?")
    print("Metric A: percentile of the deal within THAT celebrity's own row of brands (0.5 = random).")
    print("Metric B: AUC of per-brand z-scores, deals vs all non-deal cells (popularity-corrected).\n")
    rows = []
    for c, b, kind in POSITIVE_PAIRS:
        if c in M and b in M[c] and c in celebs:
            own = DOC_BY_CELEB.get(c, set())
            ref = [v for bb, v in M[c].items() if bb not in own]
            pa = pct_rank(M[c][b], ref)
            zb = Z.get(c, {}).get(b)
            rows.append((c, b, kind, M[c][b], pa, zb))
            print(f"   [{kind:<8}] {c:<20} x {b:<22} score={M[c][b]:.3f} rowPct={pa:.2f} "
                  f"z={'n/a' if zb is None else f'{zb:+.2f}'}")
    skipped = [(c, b) for c, b, _ in POSITIVE_PAIRS if not (c in M and b in M.get(c, {}))]
    print(f"\nusable positives: {len(rows)}; skipped (unresolved/unscored): {skipped}")

    def metrics(sub):
        pa = [r[4] for r in sub if r[4] is not None]
        pz = [r[5] for r in sub if r[5] is not None]
        nz = [z for c, zr in Z.items() for b, z in zr.items() if (c, b) not in DOC]
        return (statistics.mean(pa) if pa else None, auc(pz, nz), len(sub),
                sum(1 for p in pa if p >= 0.9))
    rep = {}
    for label, sub in (("ALL", rows), ("founder", [r for r in rows if r[2] == "founder"]),
                       ("endorser", [r for r in rows if r[2] == "endorser"])):
        if len(sub) >= 3:
            mA, aB, n, top = metrics(sub); rep[label] = (mA, aB)
            print(f"\n{label:<9} n={n}: mean rowPct={mA:.3f} [{verdict(mA, 0.75, 0.60)}] | "
                  f"z-AUC={aB:.3f} [{verdict(aB)}] | top-decile in row: {top}/{n}")
    mA_all, aucB_all = rep.get("ALL", (None, None))

    # ---------------- GATE 4: stability ----------------
    banner("GATE 4 - STABILITY")
    tc = celebs.get("Cedric the Entertainer") or next(iter(celebs.values()))
    tb = brands.get("Nike") or next(iter(brands.values()))
    reps = [fit_score(tc["id"], tb["id"]) for _ in range(STABILITY_REPEATS)]
    stable = all(r is not None for r in reps) and (max(reps) - min(reps)) < 1e-9
    print(f"{tc['name']} x {tb['name']}: {reps} -> {'PASS' if stable else 'FAIL'}")

    # ---------------- GATE 5: explainability ----------------
    banner("GATE 5 - EXPLAINABILITY")
    st5, d5 = qloo_get("/v2/insights", {"filter.type": "urn:entity:brand",
        "signal.interests.entities": tc["id"], "filter.results.entities": tb["id"],
        "feature.explainability": "true", "take": 1}, tag="insights[explain]")
    exp = None
    for it in (iter_result_items(d5) if st5 == 200 and d5 else []):
        exp = (it.get("query") or {}).get("explainability")
    print(f"explainability present: {bool(exp)}  {json.dumps(exp)[:300] if exp else ''}")
    gate5 = bool(exp)

    # ---------------- GATE 6: extras ----------------
    banner("GATE 6 - PITCH EXTRAS")
    extras = {}
    # 6a audiences (400'd with no params last run -> probe variants)
    aud_ok = False
    for p in ({}, {"take": 10}, {"filter.parents.types": "urn:audience:communities"},
              {"filter.parents.types": "urn:audience:global_issues"}, {"filter.query": "music"}):
        st, d = qloo_get("/v2/audiences", p, tag="audiences", quiet=True)
        print(f"6a /v2/audiences {p} -> {st}")
        if st == 200 and d: aud_ok = True; break
    extras["audiences"] = aud_ok

    # 6b demographics
    base = fit_score(tc["id"], tb["id"])
    wa = fit_score(tc["id"], tb["id"], {"signal.demographics.age": "35_and_younger"})
    wg = fit_score(tc["id"], tb["id"], {"signal.demographics.gender": "female"})
    print(f"6b demographics: base={base} +age35_and_younger={wa} +female={wg} -> "
          f"{'PASS (scores moved)' if (wa != base or wg != base) else 'WARN (no effect)'}")
    extras["demographics"] = (wa != base or wg != base)

    # 6c heatmap
    st, d = qloo_get("/v2/insights", {"filter.type": "urn:heatmap",
        "signal.interests.entities": tc["id"], "filter.location.query": "New York City"}, tag="heatmap")
    cells = ((d or {}).get("results") or {}).get("heatmap") or []
    extras["heatmap"] = st == 200 and bool(cells)
    print(f"6c heatmap: {'PASS' if extras['heatmap'] else 'FAIL'} ({len(cells)} cells)")

    # 6d compare
    cb = brands.get("Casamigos") or next(iter(brands.values()))
    st, d = qloo_get("/v2/insights/compare", {"a.signal.interests.entities": tc["id"],
        "b.signal.interests.entities": cb["id"], "take": 5}, tag="compare")
    extras["compare"] = st == 200 and bool((d or {}).get("results"))
    print(f"6d /v2/insights/compare: {'PASS' if extras['compare'] else 'FAIL/inspect'} (status {st})")

    # 6e trends via bias.trends (documented insights param)
    s1, d1 = qloo_get("/v2/insights", {"filter.type": "urn:entity:brand",
        "signal.interests.entities": tc["id"], "take": 15}, tag="trend-off")
    s2, d2 = qloo_get("/v2/insights", {"filter.type": "urn:entity:brand",
        "signal.interests.entities": tc["id"], "bias.trends": "high", "take": 15}, tag="trend-high")
    a1 = {_entity_fields(i)[0] for i in iter_result_items(d1)} if d1 else set()
    a2 = {_entity_fields(i)[0] for i in iter_result_items(d2)} if d2 else set()
    extras["trends"] = s2 == 200 and a1 != a2
    print(f"6e bias.trends: {'PASS' if extras['trends'] else 'FAIL'} (overlap {len(a1 & a2)}/15)")

    # ---------------- Cedric spotlight (from the matrix) ----------------
    banner("CEDRIC THE ENTERTAINER - SPOTLIGHT")
    if "Cedric the Entertainer" in M:
        ced = M["Cedric the Entertainer"]; cz = Z.get("Cedric the Entertainer", {})
        print("top brands by raw affinity:")
        for b, v in sorted(ced.items(), key=lambda kv: -kv[1])[:8]: print(f"   {b:<26} {v:.3f}")
        print("top brands by popularity-corrected z (distinctive to his audience):")
        for b, v in sorted(cz.items(), key=lambda kv: -kv[1])[:8]: print(f"   {b:<26} z={v:+.2f}")
        for ft, lab in (("urn:entity:destination", "destinations"), ("urn:entity:tv_show", "TV shows"),
                        ("urn:entity:place", "places")):
            st, d = qloo_get("/v2/insights", {"filter.type": ft,
                "signal.interests.entities": celebs["Cedric the Entertainer"]["id"], "take": 6}, tag=f"ced-{lab}")
            names = [_entity_fields(i)[1] for i in iter_result_items(d)] if d else []
            print(f"top {lab}: {names}")

    # ---------------- save CSVs ----------------
    with open(os.path.join(OUT_DIR, "matrix.csv"), "w") as f:
        bl = sorted(allb); f.write("celebrity," + ",".join(f'"{b}"' for b in bl) + "\n")
        for c, row in M.items():
            f.write(f'"{c}",' + ",".join(f"{row[b]:.4f}" if b in row else "" for b in bl) + "\n")
    with open(os.path.join(OUT_DIR, "positives.csv"), "w") as f:
        f.write("celebrity,brand,kind,affinity,row_pct,z\n")
        for c, b, k, s, pa, zb in rows: f.write(f'"{c}","{b}",{k},{s},{pa},{zb}\n')

    # ---------------- FINAL ----------------
    banner("FINAL REPORT")
    print(f"calls={STATS['calls']} avg latency={1000 * statistics.mean(STATS['latency']):.0f}ms "
          f"elapsed={(time.time() - t_start) / 60:.1f}min status={STATS['status']}")
    print(f"""
GATE 1 resolution ............ {'PASS' if gate1 else 'WARN'}
GATE 2 coverage / variance ... {'PASS' if gate2 and std >= 0.02 else 'FAIL'}  (coverage {100 * coverage:.0f}%, std {std:.3f})
GATE 3 discrimination ........ rowPct={mA_all if mA_all is None else round(mA_all, 3)} [{verdict(mA_all)}]  z-AUC={aucB_all if aucB_all is None else round(aucB_all, 3)} [{verdict(aucB_all)}]
GATE 4 stability ............. {'PASS' if stable else 'FAIL'}
GATE 5 explainability ........ {'PASS' if gate5 else 'FAIL'}
GATE 6 extras ................ audiences={extras['audiences']} demographics={extras['demographics']} heatmap={extras['heatmap']} compare={extras['compare']} trends={extras['trends']}
""")

main()


STEP 0 - HEALTH CHECK

--- RAW [search[health]] ---
{
  "results": [
    {
      "name": "Nike",
      "entity_id": "C70CE2B8-0AD1-4150-BF18-5A6347F2E860",
      "types": [
        "urn:entity:brand"
      ],
      "properties": {
        "akas": [
          {
            "language": "ru",
            "value": "Найк"
          },
          {
            "language": "bg",
            "value": "Найк"
          },
          {
            "language": "uk",
            "value": "Найк"
          },
          {
            "language": "es",
            "value": "Nike, Inc."
          },
          {
            "language": "es_ar",
            "value": "Nike, Inc."
          },
          {
            "language": "sr",
            "value": "Најки"
          },
          {
            "language": "hi",
            "value": "नाइकी"
          },
          {
            "language": "zh",
            "value": "耐克"
          },
          {
            "language": "ko",
            "value": "나이키"
  

In [5]:
# ===== TASTE TWIN TEST (new cell, below the big one) =====
import json, os, random, statistics, time
random.seed(7)

TWINS_PER_TYPE   = 5     # 5 people + 5 artists per celebrity
N_CONTROL_BRANDS = 25    # extra non-deal brands in the panel
N_CTRL_PER_CELEB = 10    # random non-twin controls per celebrity
os.makedirs(OUT_DIR, exist_ok=True)
CACHE = os.path.join(OUT_DIR, "twins_cache.json")
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {"twins": {}, "rows": {}, "ids": {}}
def save(): json.dump(cache, open(CACHE, "w"))

# ---- 1. brand panel (reuses resolve() cache from the big cell) ----
deal_brands = sorted({b for _, b, _ in POSITIVE_PAIRS})
names = list(dict.fromkeys(BRANDS_MAIN + deal_brands))
pool_b = [b for b in CONTROL_BRANDS if b not in names]
names += random.sample(pool_b, min(N_CONTROL_BRANDS, len(pool_b)))
panel = {}
for n in names:
    r = resolve(n, "brand")
    if r: panel[n] = r["id"]
id2name = {v: k for k, v in panel.items()}
bids = list(id2name)
probe = panel["Nike"]
print(f"panel brands resolved: {len(bids)}/{len(names)}")

# ---- 2. helpers ----
def get_cid(name):
    if name in cache["ids"]: return cache["ids"][name]
    r = resolve_signal(name, probe)
    cache["ids"][name] = r["id"] if r else None
    return cache["ids"][name]

def get_row(eid):
    if eid not in cache["rows"]:
        cache["rows"][eid] = panel_scores(eid, bids)   # {brand_id: affinity}
    return cache["rows"][eid]

def get_twins(cid, cname):
    if cid in cache["twins"]: return cache["twins"][cid]
    out, seen = [], {cid}
    for t in ("urn:entity:person", "urn:entity:artist"):
        st, d = qloo_get("/v2/insights", {"filter.type": t, "signal.interests.entities": cid,
                                          "take": 20}, tag="twins", quiet=True)
        k = 0
        for it in (iter_result_items(d) if st == 200 and d else []):
            eid, nm, _, _ = _entity_fields(it)
            if not eid or eid in seen or similarity(cname, nm) >= 0.85: continue
            seen.add(eid); out.append({"id": eid, "name": nm}); k += 1
            if k >= TWINS_PER_TYPE: break
    cache["twins"][cid] = out
    return out

def ranks(v):
    s = sorted(range(len(v)), key=lambda i: v[i]); r = [0.0] * len(v); i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and v[s[j + 1]] == v[s[i]]: j += 1
        for k in range(i, j + 1): r[s[k]] = (i + j) / 2 + 1
        i = j + 1
    return r

def spearman(a, b):
    ks = [k for k in a if k in b]
    if len(ks) < 20: return None
    x, y = ranks([a[k] for k in ks]), ranks([b[k] for k in ks])
    mx, my = statistics.mean(x), statistics.mean(y)
    num = sum((p - mx) * (q - my) for p, q in zip(x, y))
    den = (sum((p - mx) ** 2 for p in x) * sum((q - my) ** 2 for q in y)) ** 0.5
    return num / den if den else None

def pct_in_row(row, bid):
    if bid not in row: return None
    ref = [v for k, v in row.items() if k != bid]
    return pct_rank(row[bid], ref) if len(ref) >= 20 else None

def boot(x, n=2000):
    m = sorted(statistics.mean(random.choices(x, k=len(x))) for _ in range(n))
    return m[int(.025 * n)], m[int(.975 * n)]

# ---- 3. collect (resumable) ----
banner("TWIN TEST - COLLECTING")
t0 = time.time()
deal_celebs = sorted({c for c, _, _ in POSITIVE_PAIRS})
info = {}
for i, cname in enumerate(deal_celebs, 1):
    cid = get_cid(cname)
    if not cid: continue
    row = get_row(cid)
    tw = []
    for t in get_twins(cid, cname):
        r = get_row(t["id"])
        if len(r) >= 20: tw.append({**t, "row": r})
    info[cname] = {"cid": cid, "row": row, "twins": tw}
    if i % 5 == 0:
        save(); print(f"  {i}/{len(deal_celebs)} celebs | {(time.time()-t0)/60:.1f} min | calls {STATS['calls']}")
save()

# control pool = every twin row, owner tracked
pool = [(c, t["id"], t["row"]) for c, v in info.items() for t in v["twins"]]
def controls_for(cname, k):
    mine = {t["id"] for t in info[cname]["twins"]}
    cand = [p for p in pool if p[0] != cname and p[1] not in mine]
    return random.sample(cand, min(k, len(cand)))

# ---- 4. sample-size report ----
banner("SAMPLE SIZE")
ntw = [len(v["twins"]) for v in info.values()]
print(f"celebrities usable: {len(info)}/{len(deal_celebs)}  (need >=40)")
print(f"twins per celebrity: median {statistics.median(ntw)}, min {min(ntw)}, total twin rows {len(pool)}")

# ---- 5. TEST A: do twins have similar brand taste as the celebrity? ----
banner("TEST A - TWIN BRAND-TASTE AGREEMENT (Spearman over panel brands)")
dA, tw_m, ct_m = [], [], []
for cname, v in info.items():
    t = [spearman(v["row"], x["row"]) for x in v["twins"]]
    c = [spearman(v["row"], p[2]) for p in controls_for(cname, N_CTRL_PER_CELEB)]
    t, c = [x for x in t if x is not None], [x for x in c if x is not None]
    if t and c:
        tw_m.append(statistics.mean(t)); ct_m.append(statistics.mean(c)); dA.append(tw_m[-1] - ct_m[-1])
loA, hiA = boot(dA)
print(f"celebs n={len(dA)} | twin rho {statistics.mean(tw_m):.3f} vs control rho {statistics.mean(ct_m):.3f}")
print(f"diff {statistics.mean(dA):+.3f} 95% [{loA:+.3f}, {hiA:+.3f}] | twins beat controls for {sum(d>0 for d in dA)}/{len(dA)} celebs")

# ---- 6. TEST B: does a documented deal transfer to twins? ----
banner("TEST B - DOES THE DEAL BRAND RANK HIGH FOR THE TWINS TOO?")
rowsB = []
for c, b, kind in POSITIVE_PAIRS:
    if c not in info or b not in panel: continue
    bid = panel[b]; v = info[c]
    own = pct_in_row(v["row"], bid)
    tp = [(pct_in_row(x["row"], bid), x["name"]) for x in v["twins"]]
    tp = [(p, n) for p, n in tp if p is not None]
    cp = [pct_in_row(p[2], bid) for p in controls_for(c, len(v["twins"]) or N_CTRL_PER_CELEB)]
    cp = [p for p in cp if p is not None]
    if own is None or not tp or not cp: continue
    rowsB.append({"c": c, "b": b, "own": own, "twin": statistics.mean(p for p, _ in tp),
                  "ctrl": statistics.mean(cp), "best_t": max(p for p, _ in tp), "best_c": max(cp), "tp": tp})
dB = [r["twin"] - r["ctrl"] for r in rowsB]
loB, hiB = boot(dB)
print(f"deals n={len(rowsB)}  (need >=30)")
print(f"mean percentile of deal brand: celebrity {statistics.mean(r['own'] for r in rowsB):.3f} | "
      f"twins {statistics.mean(r['twin'] for r in rowsB):.3f} | random controls {statistics.mean(r['ctrl'] for r in rowsB):.3f}")
print(f"twin - control {statistics.mean(dB):+.3f} 95% [{loB:+.3f}, {hiB:+.3f}] | twins beat controls on {sum(d>0 for d in dB)}/{len(dB)} deals")
print(f"at least one twin in top 10% of its row: {sum(r['best_t']>=.9 for r in rowsB)}/{len(rowsB)} deals "
      f"vs controls {sum(r['best_c']>=.9 for r in rowsB)}/{len(rowsB)}")

# ---- 7. demo examples ----
banner("EXAMPLES  (celebrity x brand -> best twins for that brand)")
for c, b in [("Serena Williams", "Nike"), ("LeBron James", "Nike"), ("Ryan Reynolds", "Aviation American Gin"),
             ("Jennifer Aniston", "Aveeno"), ("Kevin Hart", "Gran Coramino"), ("Taylor Swift", "Diet Coke")]:
    r = next((x for x in rowsB if x["c"] == c and x["b"] == b), None)
    if r:
        top = sorted(r["tp"], reverse=True)[:3]
        print(f"{c} x {b}: own pct {r['own']:.2f} | twins -> " + ", ".join(f"{n} {p:.2f}" for p, n in top))

# ---- 8. verdict ----
banner("VERDICT")
okA, okB = loA > 0, loB > 0
print(f"Test A (taste agreement beats controls): {'PASS' if okA else 'FAIL'}")
print(f"Test B (deal transfers to twins):        {'PASS' if okB else 'FAIL'}")


panel brands resolved: 99/135

TWIN TEST - COLLECTING

--- RAW [twins] ---
{
  "success": true,
  "results": {
    "entities": [
      {
        "name": "Austin Carlile",
        "entity_id": "48FB7A40-FF5A-4530-BF7C-97FDCB644078",
        "type": "urn:entity",
        "subtype": "urn:entity:person",
        "properties": {
          "akas": [
            {
              "value": "Austin Carlile",
              "language": "sq"
            },
            {
              "value": "אוסטין קרלייל",
              "language": "he"
            },
            {
              "value": "Austin Carlile",
              "language": "sl"
            },
            {
              "value": "Austin Carlile",
              "language": "pt"
            },
            {
              "value": "Austin Carlile",
              "language": "nb"
            },
            {
              "value": "Austin Robert Carlile",
              "language": "fr"
            },
            {
              "value": "Ости